In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D, Dropout
import numpy as np

In [ ]:
train_dir = r"C:\Users\User\Desktop\EJAZTECH.AI-Mentorship-Program\Image classification\Butterfly image data\train"
val_dir = r"C:\Users\User\Desktop\EJAZTECH.AI-Mentorship-Program\Image classification\Butterfly image data\valid"
test_dir = r"C:\Users\User\Desktop\EJAZTECH.AI-Mentorship-Program\Image classification\Butterfly image data\test"

In [ ]:
# Checking to see it works
print("train...")
train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,               # We use the variable you just created!
    image_size=(224, 224),   # Squeezing the animals into neat squares
    batch_size=32
)

print("valid...")
valid_data = tf.keras.utils.image_dataset_from_directory(
    val_dir,                 # We use your validation variable!
    image_size=(224, 224),
    batch_size=32
)

print("test...")
test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False   
)

In [ ]:
for folder in os.listdir(train_dir):
    folder_path = os.path.join(train_dir, folder)
    image_count = 0
    
    for file in os.listdir(folder_path):
        image_path = os.path.join(folder_path, file)
        
        # Show the image on the screen
        plt.figure(figsize=(2, 2))
        plt.imshow(plt.imread(image_path))
        plt.show()
        
        image_count += 1
        # Stop after 5 pictures per animal
        if image_count == 5:
            break

In [ ]:
class_labels =[entry for entry in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, entry))]

print("Class labels:")

# 2. Now the name here perfectly matches line 1
for label in class_labels:
    label_path = os.path.join(train_dir, label)
    
    # 3. Fixed os.list to os.listdir
    # 4. Spelled it num_contents with an 'm'
    num_contents = len(os.listdir(label_path)) 
    
    # 5. Changed F to f, and used squiggly brackets {label}
    print(f"{label}: {num_contents} items")

In [ ]:
butterfly_count_dict = {}

# 1. Look inside the train_dir you already created in Cell [2]
for folder in os.listdir(train_dir):
    folder_path = os.path.join(train_dir, folder)
    
    # We just want to make sure it's a folder, not a hidden file
    if os.path.isdir(folder_path):
        # Shortcut! len() automatically counts everything inside the folder
        butterfly_count = len(os.listdir(folder_path))
        butterfly_count_dict[folder] = butterfly_count

# 2. Get the labels and numbers ready
butterfly_subfolders = list(butterfly_count_dict.keys())
butterfly_counts = list(butterfly_count_dict.values())

# 3. Draw the Giant Bar Chart!
# I changed figsize to (20, 6) to make the chart SUPER wide to fit 100 butterflies
plt.figure(figsize=(20, 6)) 
plt.bar(butterfly_subfolders, butterfly_counts, color='skyblue')

plt.xlabel("Butterfly Species", fontsize=14)
plt.ylabel("Number of Images", fontsize=14)
plt.title("Image Counts for All 100 Butterfly Species!", fontsize=18)

# I rotated the names 90 degrees (straight up) and made the font size 8 so they fit!
plt.xticks(rotation=90, fontsize=8) 

# Show the masterpiece
plt.show()

In [ ]:
butterfly_val_dict = {}

# 1. We use val_dir this time!
for folder in os.listdir(val_dir):
    folder_path = os.path.join(val_dir, folder)
    
    if os.path.isdir(folder_path):
        val_count = len(os.listdir(folder_path))
        butterfly_val_dict[folder] = val_count

# 2. Get the labels and numbers ready
val_subfolders = list(butterfly_val_dict.keys())
val_counts = list(butterfly_val_dict.values())

# 3. Draw the Validation Bar Chart!
plt.figure(figsize=(20, 6)) # Super wide for 100 butterflies!
# Let's make it green so we know it's the Validation data!
plt.bar(val_subfolders, val_counts, color='lightgreen') 

plt.xlabel("Butterfly Species", fontsize=14)
plt.ylabel("Number of Images", fontsize=14)
plt.title("Image Counts for Butterfly Validation Data", fontsize=18)

plt.xticks(rotation=90, fontsize=8) 
plt.show()

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomTranslation, Rescaling

data_augmentation = tf.keras.Sequential([
    # 1. Your brilliant catch: Rescaling the pixels to make math easier!
    Rescaling(1./255),
    
    # 2. Flipping horizontally AND vertically (just like your code!)
    RandomFlip("horizontal_and_vertical"), 
    
    # 3. Rotating by 20% (Your rotation_range = 20)
    RandomRotation(0.2), 
    
    # 4. Shifting the picture up/down and left/right (Your width_shift and height_shift)
    RandomTranslation(height_factor=0.2, width_factor=0.2),
    
    # 5. Zooming in by 20% (Your zoom_range = 0.2)
    RandomZoom(0.2), 
])

In [ ]:
#Transfer learning
img_size = (224, 224)
# This adds the 3 colors (RGB) to make (224, 224, 3)
image_shape = img_size + (3, ) 

# Hiring the smart brain!
mobilenet = tf.keras.applications.MobileNetV2(
    input_shape = image_shape, 
    include_top = False,       # Chopping off its old guessing layer
    weights = 'imagenet'      
)

In [ ]:
mobilenet.summary()

In [ ]:
mobilenet.trainable = False
mobilenet.summary()

In [ ]:
# Make sure the brain is frozen
mobilenet.trainable = False

model = Sequential([
    # Add your data_augmentation here if you have the variable!
    mobilenet,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(100, activation='softmax')
])
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# Put the glasses on all three datasets
train_data = train_data.map(lambda x, y: (preprocess_input(x), y))
valid_data = valid_data.map(lambda x, y: (preprocess_input(x), y))
test_data = test_data.map(lambda x, y: (preprocess_input(x), y))

In [ ]:
# 1. Setup the Brain
mobilenet = tf.keras.applications.MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')

# 2. UNFREEZE a little bit right away (The top 20 layers)
mobilenet.trainable = True
for layer in mobilenet.layers[:-20]:
    layer.trainable = False

# 3. Build the Model
model = Sequential([
    mobilenet,
    GlobalAveragePooling2D(),
    Dropout(0.2),              # This helps stop the "memorizing" (Overfitting)
    Dense(128, activation='relu'),
    Dense(100, activation='softmax') # 100 Butterflies
])

# 4. Compile with a SLOW Learning Rate (This is the secret sauce)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Train for 10 Epochs
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10
)

In [ ]:
#How Model performs on test data
results = model.evaluate(test_data)
print(f"Test Loss: {results[0]}")
print(f"Test Accuracy: {results[1]*100:.2f}%")

In [ ]:
import os

# 1. Let's go back to the original folder and find
# (Use train_dir because it has all 100 butterfly folders)
class_labels = sorted(os.listdir(train_dir)) 

# 2. Use the test data
dataset_to_use = test_data 

# 3. Take one batch and show the pictures
for images, labels in dataset_to_use.take(1):
    predictions = model.predict(images)
    predicted_labels = np.argmax(predictions, axis=1)

    plt.figure(figsize=(12, 12))
    num_to_show = min(9, len(images)) 
    
    for i in range(num_to_show):
        plt.subplot(3, 3, i + 1)
        
        # Make the picture look normal for humans 
        img_display = (images[i].numpy() + 1) / 2 
        plt.imshow(img_display) 
        
        # Get the name of the Robot's guess
        predicted_name = class_labels[predicted_labels[i]]
        
        # Get the real answer 
        # (Using labels[i] because it's a number 0-99)
        real_label_index = int(labels[i])
        real_name = class_labels[real_label_index]
            
        # Green for right, Red for wrong
        color = 'green' if predicted_name == real_name else 'red'
        plt.title(f"Guess: {predicted_name}\nReal: {real_name}", color=color, fontsize=10)
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

In [ ]:
# Save Butterfly Model
model_butterfly.save("butterfly_expert_95pct.keras")
print("Butterfly classification")